<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Local density: LOF

**[Local density: LOF](https://learning.nextia-ai.com/courses/anomaly-detection/m04/local-density-lof/)** · Anomaly Detection: Find What Is Unusual · Module 4, lesson 2 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score a point against the density of its own area with the Local Outlier Factor, see where that beats a plain distance, and measure honestly where it fails on Larkfield's payments.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. The notebook takes about a minute to run. |
| **You should know** | kNN distance and scaling, from [Far from the neighbours: kNN distance](https://learning.nextia-ai.com/courses/anomaly-detection/m04/far-from-the-neighbours-knn-distance/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m04/local-density-lof/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M04-L02-local-density-lof/local-density-lof-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Far, or far for this area? A tiny example

In a city centre, houses stand 10 metres apart. In the countryside, farms stand 500 metres apart. A house 200 metres from the centre is strange. A farm 200 metres from the next farm is normal.

The **Local Outlier Factor** (LOF) compares a point's distance to its neighbours with the neighbours' own distances to *their* neighbours. About 1 means "as crowded as my neighbours". Much more than 1 means "far for this area".

Run the next cell. It builds a dense group d1 to d9 (0.2 apart), a spread-out group s1 to s9 (1.5 apart), and two test points: P, 0.8 from the dense group, and Q, 1.5 from the spread-out group. It scores every point with kNN distance (k = 3) and with LOF (k = 3).

> **Check.** You should see P with a kNN distance of 0.82 and an LOF of 3.46. kNN gives its highest score to the four corners of the spread-out group and to Q; LOF gives its highest score to P.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
dense = [[1.0 + 0.2 * i, 1.0 + 0.2 * j] for i in range(3) for j in range(3)]     # 9 points, 0.2 apart
spread = [[6.0 + 1.5 * i, 2.0 + 1.5 * j] for i in range(3) for j in range(3)]    # 9 points, 1.5 apart
points = np.array(dense + spread + [[2.2, 1.2], [10.5, 3.5]])                     # then P and Q
names = [f"d{i}" for i in range(1, 10)] + [f"s{i}" for i in range(1, 10)] + ["P", "Q"]
k = 3
dist, _ = NearestNeighbors(n_neighbors=k + 1).fit(points).kneighbors(points)     # +1: the point itself
lof = LocalOutlierFactor(n_neighbors=k).fit(points)
tiny = pd.DataFrame({"kNN distance": dist[:, k], "LOF": -lof.negative_outlier_factor_}, index=names).round(2)
print(tiny.loc[["d1", "d5", "s1", "s5", "P", "Q"]])
top = tiny["kNN distance"].max()
print("highest kNN distance:", top, "for", tiny.index[tiny["kNN distance"] == top].tolist())
print("highest LOF:", tiny["LOF"].max(), "for", tiny["LOF"].idxmax(), "| all the others:", tiny["LOF"].drop("P").min(),
      "to", tiny["LOF"].drop("P").max())

## 2. Larkfield: two kinds of account

Personal and business accounts spend very differently. Run the next cell. It loads the payments, the raw and the behaviour features, and compares the two kinds of account in the reference months.

> **Check.** You should see a median of 34.32 dollars for personal accounts and 257.50 for business accounts.

A payment of 400 dollars is rare for a personal account and common for a business account. One global distance cannot see that; a local density can, in principle.

In [ ]:
import compare, features
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
raw = features.raw(payments)
raw["amount"] = np.log(raw.amount)
behaviour = features.behaviour(payments)
amounts = reference.groupby("account_type").amount
print(amounts.median().round(2).to_string())
print("share of payments of 400 dollars or more:")
print(amounts.apply(lambda a: (a >= 400).mean()).round(4).to_string())

## 3. Run LOF on the payments

The next cell defines `lof_scores()`. It scales the features on the reference months, stores a sample of 20,000 reference payments, and scores each evaluation payment against them. `novelty=True` tells scikit-learn that you score **new** points, not the points you trained on. It adds two rows to the comparison table. It takes about 15 seconds.

> **Predict.** In the last lesson, kNN distance gave 0.219 (raw) and 0.505 (behaviour). Will LOF do better or worse?

> **Check.** You should see 0.168 for the raw features and 0.082 for the behaviour features.

In [ ]:
def scaled(X, sample=20_000, seed=0):
    """Scale on the reference months; return a reference sample and the evaluation rows."""
    scaler = StandardScaler().fit(X.loc[reference.index])
    ref = scaler.transform(X.loc[reference.index])
    ref = ref[np.random.default_rng(seed).choice(len(ref), sample, replace=False)]
    return ref, scaler.transform(X.loc[evaluation.index])

def lof_scores(X, k=35):
    """LOF of each evaluation payment against a sample of 20,000 reference payments. Higher = more unusual."""
    ref, ev = scaled(X)
    model = LocalOutlierFactor(n_neighbors=k, novelty=True).fit(ref)   # novelty=True: score new points
    return -model.score_samples(ev)

table = compare.Table()
table.add(compare.score_payments(evaluation, lof_scores(raw), "LOF", "raw features, 35 neighbours", explain_minutes=3))
table.add(compare.score_payments(evaluation, lof_scores(behaviour), "LOF", "behaviour features, 35 neighbours",
                                 explain_minutes=3))
table.show()

## 4. Why does LOF fail here?

With the behaviour features, LOF finds far fewer planted payments than kNN distance. Look at the recall of each kind of fraud. Run the next cell.

> **Check.** You should see a recall of 0.060 for card testing.

In [ ]:
lof_behaviour = lof_scores(behaviour)
print(compare.recall_by_kind(evaluation, lof_behaviour).round(3).to_string())

Card testing is almost invisible to LOF. Now look at the neighbours. The next cell finds the 35 nearest reference neighbours of every evaluation payment and counts how many of them are planted payments. (It uses the course's answer key through `compare.planted_payments()`. You can unpack the key yourself with `compare.reveal()`: it is there to score your work, not to keep a secret.)

> **Check.** You should see about 0.95 for card testing and 0.002 for honest payments.

In [ ]:
ref, ev = scaled(behaviour)
ref_ids = reference.payment_id.to_numpy()[np.random.default_rng(0).choice(len(reference), 20_000, replace=False)]
kinds = compare.planted_payments().set_index("payment_id").kind
ref_planted = pd.Series(ref_ids).isin(kinds.index).to_numpy()
_, idx = NearestNeighbors(n_neighbors=35).fit(ref).kneighbors(ev)
ev_kind = evaluation.payment_id.map(kinds).fillna("honest").to_numpy()
share = {kind: round(float(ref_planted[idx[ev_kind == kind]].mean()), 3) for kind in ["card_testing", "stolen_card",
                                                                              "account_takeover", "honest"]}
print("share of planted payments among the 35 nearest reference neighbours:", share)

A card test's neighbours are mostly **old card tests** of the reference months. They form a small, dense crowd of their own. Inside that crowd, a new card test is as crowded as its neighbours, so its LOF is about 1: "normal for this area". LOF does exactly what it promises, and here that is the wrong question. kNN distance, which asks "far from the normal crowd?", does not have this problem with k = 10.

## 5. Check yourself

**Your turn 1.** In the tiny example, what is P's LOF? Write it as `lof_p` (2 decimals).

In [ ]:
lof_p = 3.46

In [ ]:
expect('lof_p', lof_p, 3.46, tolerance=0.01)

**Your turn 2.** Which kind of planted fraud has the lowest recall with LOF on the behaviour features? Write its name, as `recall_by_kind` prints it, in `worst_kind`.

In [ ]:
worst_kind = compare.recall_by_kind(evaluation, lof_behaviour).idxmin()
print(worst_kind)

In [ ]:
expect_hash('worst_kind', worst_kind, 'a6f243ed80994ad7')

**Your turn 3.** A larger neighbourhood looks past a small crowd of old frauds. Score the behaviour features with 100 neighbours, and put the precision at the budget in `lof_100`. Predict first: better or worse than 35 neighbours? This cell takes about 20 seconds.

In [ ]:
lof_100 = compare.score_payments(evaluation, lof_scores(behaviour, k=100), "LOF",
                                 "behaviour features, 100 neighbours")["precision at budget"]
print(lof_100)

In [ ]:
expect('lof_100', lof_100, 0.154, tolerance=0.001)

In the solution: 0.154, better than 0.082 with 35 neighbours, and still far below kNN distance (0.505).

## Recap

- LOF compares a point's distance to its neighbours with their own distances: about 1 is normal for the area, much more than 1 is strange **for this area**.
- It beats a plain distance when areas have different densities: in the tiny example, it finds P and ignores the spread-out group's corners.
- On Larkfield's behaviour features, it fails (0.082): card tests form a dense crowd with old card tests, so they look normal locally (0.060 recall).

The next lesson turns the question around: instead of measuring distances, cut the data at random and count how fast a point is alone.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m04/local-density-lof/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.